# Bharat Herald Data Cleaning & Analytics Pipeline

This notebook performs the following operations:
1. Extracts raw datasets from the `public` schema in the `bharat_herald` database.
2. Cleans and normalizes the datasets.
3. Loads cleaned datasets into the `cleaned_data` schema.
4. Runs analytical queries and saves results into the `analytics` schema.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Database connection details
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "bharat_herald"
DB_USER = "postgres"
DB_PASS = "postgresql"

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Engine created successfully.")

In [ ]:
# Import raw datasets from public schema
print("Importing raw datasets from public schema...")
with engine.begin() as conn:
    df_city = pd.read_sql_table("dim_city", conn, schema="public")
    df_cat = pd.read_sql_table("dim_ad_category", conn, schema="public")
    df_print = pd.read_sql_table("fact_print_sales", conn, schema="public")
    df_rev = pd.read_sql_table("fact_ad_revenue", conn, schema="public")
    df_read = pd.read_sql_table("fact_city_readiness", conn, schema="public")
    df_pilot = pd.read_sql_table("fact_digital_pilot", conn, schema="public")

print(f"Loaded city shape: {df_city.shape}")
print(f"Loaded category shape: {df_cat.shape}")
print(f"Loaded print sales shape: {df_print.shape}")
print(f"Loaded ad revenue shape: {df_rev.shape}")
print(f"Loaded readiness shape: {df_read.shape}")
print(f"Loaded pilot shape: {df_pilot.shape}")

In [ ]:
# Define cleaning functions
def clean_numeric_int(val):
    if pd.isna(val):
        return 0
    if isinstance(val, (int, float)):
        return int(val)
    val_str = str(val).strip()
    val_str = re.sub(r'[^\d\-]', '', val_str)
    if val_str == '':
        return 0
    return int(val_str)

def clean_numeric_float(val):
    if pd.isna(val):
        return 0.0
    if isinstance(val, (int, float)):
        return float(val)
    val_str = str(val).strip()
    val_str = re.sub(r'[^\d\.\-]', '', val_str)
    if val_str == '':
        return 0.0
    try:
        return float(val_str)
    except ValueError:
        return 0.0

def normalize_quarter(q):
    if not isinstance(q, str):
        return q
    q = q.strip()
    if re.match(r'^\d{4}-Q[1-4]$', q):
        return q
    m = re.match(r'^Q([1-4])-(\d{4})$', q)
    if m:
        return f"{m.group(2)}-Q{m.group(1)}"
    m = re.match(r'^(\d)th\s+Qtr\s+(\d{4})$', q)
    if m:
        return f"{m.group(2)}-Q{m.group(1)}"
    return q

def normalize_currency_val(revenue, currency):
    curr = str(currency).strip().upper()
    if curr == 'USD':
        return revenue * 80.0
    elif curr == 'EUR':
        return revenue * 85.0
    elif curr in ['INR', 'IN RUPEES']:
        return revenue * 1.0
    else:
        return revenue

print("Cleaning functions defined.")

In [ ]:
# Clean dim_city and dim_ad_category
print("Cleaning dim_city...")
df_city_clean = df_city.copy()
df_city_clean['city'] = df_city_clean['city'].str.strip().str.title()
df_city_clean['state'] = df_city_clean['state'].str.strip().str.title()
state_map = {
    'Delhi': 'Delhi',
    'Delhi ': 'Delhi',
    'Uttar-Pradesh': 'Uttar Pradesh',
    'Madhya_Pradesh': 'Madhya Pradesh'
}
df_city_clean['state'] = df_city_clean['state'].replace(state_map)
df_city_clean['tier'] = df_city_clean['tier'].str.strip()

print("Cleaning dim_ad_category...")
df_cat_clean = df_cat.copy()
df_cat_clean['standard_ad_category'] = df_cat_clean['standard_ad_category'].str.strip()
df_cat_clean['category_group'] = df_cat_clean['category_group'].str.strip()
df_cat_clean['example_brands'] = df_cat_clean['example_brands'].str.strip()
print("Dimension tables cleaned.")

In [ ]:
# Clean fact_print_sales
print("Cleaning fact_print_sales...")
df_print_clean = df_print.copy()
df_print_clean['language'] = df_print_clean['language'].str.strip().str.title()
df_print_clean['state'] = df_print_clean['state'].str.strip().str.title().replace(state_map)

# Enforce net_circulation = copies_printed - copies_returned
df_print_clean['net_circulation'] = df_print_clean['copies_printed'] - df_print_clean['copies_returned']
df_print_clean['net_circulation'] = df_print_clean['net_circulation'].clip(lower=0)

# Deduplicate key dimensions
df_print_clean = df_print_clean.drop_duplicates(subset=['edition_id', 'city_id', 'language', 'state', 'month'])
print(f"Cleaned print sales shape: {df_print_clean.shape}")

In [ ]:
# Clean fact_ad_revenue
print("Cleaning fact_ad_revenue...")
df_rev_clean = df_rev.copy()

# Group by the dimensional keys and sum the ad revenues to consolidate duplicates
df_rev_grouped = df_rev_clean.groupby(['edition_id', 'city_id', 'ad_category_id', 'quarter'], as_index=False).agg({
    'raw_ad_revenue': 'sum',
    'ad_revenue_in_inr': 'sum',
    'comments': lambda x: '; '.join(str(v) for v in x.dropna() if str(v).strip())
})
df_rev_grouped['comments'] = df_rev_grouped['comments'].replace('', None)
df_rev_grouped['currency'] = 'INR'

df_rev_final = df_rev_grouped[['edition_id', 'city_id', 'ad_category_id', 'quarter', 'raw_ad_revenue', 'currency', 'ad_revenue_in_inr', 'comments']]
print(f"Cleaned ad revenue shape: {df_rev_final.shape} (consolidated from {df_rev_clean.shape})")

In [ ]:
# Clean fact_city_readiness and fact_digital_pilot
print("Cleaning fact_city_readiness...")
df_read_clean = df_read.copy()

print("Cleaning fact_digital_pilot...")
df_pilot_clean = df_pilot.copy()

print(f"Readiness shape: {df_read_clean.shape}, Pilot shape: {df_pilot_clean.shape}")

In [ ]:
# Create cleaned_data schema and import datasets
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS cleaned_data;"))
    print("Schema cleaned_data created or verified.")

print("Writing clean datasets to cleaned_data schema...")
df_city_clean.to_sql("dim_city", engine, schema="cleaned_data", if_exists="replace", index=False)
df_cat_clean.to_sql("dim_ad_category", engine, schema="cleaned_data", if_exists="replace", index=False)
df_print_clean.to_sql("fact_print_sales", engine, schema="cleaned_data", if_exists="replace", index=False)
df_rev_final.to_sql("fact_ad_revenue", engine, schema="cleaned_data", if_exists="replace", index=False)
df_read_clean.to_sql("fact_city_readiness", engine, schema="cleaned_data", if_exists="replace", index=False)
df_pilot_clean.to_sql("fact_digital_pilot", engine, schema="cleaned_data", if_exists="replace", index=False)

print("Data loaded into cleaned_data schema successfully.")

In [ ]:
# Create analytics schema
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS analytics;"))
    print("Schema analytics created or verified.")

In [ ]:
# Define and run analytical queries
queries = {
    "business_request_1": """
        WITH monthly_circulation AS (
            SELECT
                c.city AS city_name,
                to_char(p.month, 'YYYY-MM') AS month_str,
                p.month,
                p.net_circulation,
                LAG(p.net_circulation) OVER (PARTITION BY p.city_id ORDER BY p.month) AS prev_net_circulation
            FROM cleaned_data.fact_print_sales p
            JOIN cleaned_data.dim_city c ON p.city_id = c.city_id
        ),
        declines AS (
            SELECT
                city_name,
                month_str,
                net_circulation,
                prev_net_circulation,
                (net_circulation - prev_net_circulation) AS circulation_change
            FROM monthly_circulation
            WHERE prev_net_circulation IS NOT NULL
        )
        SELECT
            city_name,
            month_str AS month,
            net_circulation,
            circulation_change AS decline_value
        FROM declines
        ORDER BY circulation_change ASC
        LIMIT 3;
    """,
    "business_request_2": """
        WITH yearly_category_rev AS (
            SELECT
                substring(r.quarter from 1 for 4)::int AS year,
                cat.standard_ad_category AS category_name,
                SUM(r.ad_revenue_in_inr) AS category_revenue
            FROM cleaned_data.fact_ad_revenue r
            JOIN cleaned_data.dim_ad_category cat ON r.ad_category_id = cat.ad_category_id
            GROUP BY 1, 2
        ),
        yearly_total_rev AS (
            SELECT
                substring(r.quarter from 1 for 4)::int AS year,
                SUM(r.ad_revenue_in_inr) AS total_revenue_year
            FROM cleaned_data.fact_ad_revenue r
            GROUP BY 1
        )
        SELECT
            cr.year,
            cr.category_name,
            cr.category_revenue,
            tr.total_revenue_year,
            ROUND(((cr.category_revenue / tr.total_revenue_year) * 100)::numeric, 2) AS pct_of_year_total
        FROM yearly_category_rev cr
        JOIN yearly_total_rev tr ON cr.year = tr.year
        WHERE (cr.category_revenue / tr.total_revenue_year) > 0.50
        ORDER BY cr.year ASC, pct_of_year_total DESC;
    """,
    "business_request_3": """
        WITH city_2024_print AS (
            SELECT
                c.city AS city_name,
                SUM(p.copies_printed) AS copies_printed_2024,
                SUM(p.net_circulation) AS net_circulation_2024
            FROM cleaned_data.fact_print_sales p
            JOIN cleaned_data.dim_city c ON p.city_id = c.city_id
            WHERE extract(year from p.month) = 2024
            GROUP BY c.city
        )
        SELECT
            city_name,
            copies_printed_2024,
            net_circulation_2024,
            ROUND((net_circulation_2024::numeric / copies_printed_2024), 4) AS efficiency_ratio,
            RANK() OVER (ORDER BY (net_circulation_2024::numeric / copies_printed_2024) DESC) AS efficiency_rank_2024
        FROM city_2024_print
        ORDER BY efficiency_ratio DESC
        LIMIT 5;
    """,
    "business_request_4": """
        WITH q1_2021 AS (
            SELECT city_id, internet_penetration AS internet_rate_q1_2021
            FROM cleaned_data.fact_city_readiness
            WHERE quarter = '2021-Q1'
        ),
        q4_2021 AS ( 
            SELECT city_id, internet_penetration AS internet_rate_q4_2021
            FROM cleaned_data.fact_city_readiness
            WHERE quarter = '2021-Q4'
        )
        SELECT
            c.city AS city_name,
            q1.internet_rate_q1_2021,
            q4.internet_rate_q4_2021,
            (q4.internet_rate_q4_2021 - q1.internet_rate_q1_2021) AS delta_internet_rate
        FROM q1_2021 q1
        JOIN q4_2021 q4 ON q1.city_id = q4.city_id
        JOIN cleaned_data.dim_city c ON q1.city_id = c.city_id
        ORDER BY delta_internet_rate DESC;
    """,
    "business_request_5": """
        WITH yearly_metrics AS (
            SELECT
                c.city_id,
                c.city AS city_name,
                year_val,
                SUM(net_circulation) AS yearly_net_circulation,
                SUM(ad_revenue) AS yearly_ad_revenue
            FROM (
                SELECT city_id, extract(year from month)::int AS year_val, net_circulation, 0::numeric AS ad_revenue
                FROM cleaned_data.fact_print_sales
                UNION ALL
                SELECT city_id, substring(quarter from 1 for 4)::int AS year_val, 0 AS net_circulation, ad_revenue_in_inr AS ad_revenue
                FROM cleaned_data.fact_ad_revenue
            ) t
            JOIN cleaned_data.dim_city c ON t.city_id = c.city_id
            WHERE year_val BETWEEN 2019 AND 2024
            GROUP BY c.city_id, c.city, year_val
        ),
        yoy_diffs AS (
            SELECT
                city_id,
                city_name,
                year_val,
                yearly_net_circulation,
                yearly_ad_revenue,
                LAG(yearly_net_circulation) OVER (PARTITION BY city_id ORDER BY year_val) AS prev_circ,
                LAG(yearly_ad_revenue) OVER (PARTITION BY city_id ORDER BY year_val) AS prev_rev
            FROM yearly_metrics
        ),
        decreases AS (
            SELECT
                city_id,
                city_name,
                year_val,
                yearly_net_circulation,
                yearly_ad_revenue,
                (CASE WHEN year_val = 2019 THEN TRUE ELSE yearly_net_circulation < prev_circ END) AS circ_decreased,
                (CASE WHEN year_val = 2019 THEN TRUE ELSE yearly_ad_revenue < prev_rev END) AS rev_decreased
            FROM yoy_diffs
        ),
        city_summary AS (
            SELECT
                city_id,
                city_name,
                (COUNT(CASE WHEN year_val > 2019 AND circ_decreased THEN 1 END) = 5) AS is_declining_print,
                (COUNT(CASE WHEN year_val > 2019 AND rev_decreased THEN 1 END) = 5) AS is_declining_ad_revenue
            FROM decreases
            GROUP BY city_id, city_name
        )
        SELECT
            m.city_name,
            m.year_val AS year,
            m.yearly_net_circulation,
            m.yearly_ad_revenue,
            (CASE WHEN cs.is_declining_print THEN 'Yes' ELSE 'No' END) AS is_declining_print,
            (CASE WHEN cs.is_declining_ad_revenue THEN 'Yes' ELSE 'No' END) AS is_declining_ad_revenue,
            (CASE WHEN cs.is_declining_print AND cs.is_declining_ad_revenue THEN 'Yes' ELSE 'No' END) AS is_declining_both
        FROM yearly_metrics m
        JOIN city_summary cs ON m.city_id = cs.city_id
        ORDER BY cs.is_declining_print DESC, cs.is_declining_ad_revenue DESC, m.city_name, m.year_val;
    """,
    "business_request_6": """
        WITH readiness_2021 AS (
            SELECT
                city_id,
                AVG((literacy_rate + smartphone_penetration + internet_penetration) / 3.0) AS readiness_score_2021
            FROM cleaned_data.fact_city_readiness
            WHERE quarter LIKE '2021%%'
            GROUP BY city_id
        ),
        pilot_engagement_2021 AS (
            SELECT
                city_id,
                SUM(downloads_or_accesses)::numeric / SUM(users_reached) AS engagement_metric_2021
            FROM cleaned_data.fact_digital_pilot
            WHERE launch_month LIKE '2021%%'
            GROUP BY city_id
        ),
        ranks AS (
            SELECT
                c.city AS city_name,
                r.readiness_score_2021,
                pe.engagement_metric_2021,
                RANK() OVER (ORDER BY r.readiness_score_2021 DESC) AS readiness_rank_desc,
                RANK() OVER (ORDER BY pe.engagement_metric_2021 ASC) AS engagement_rank_asc
            FROM readiness_2021 r
            JOIN pilot_engagement_2021 pe ON r.city_id = pe.city_id
            JOIN cleaned_data.dim_city c ON r.city_id = c.city_id
        )
        SELECT
            city_name,
            ROUND(readiness_score_2021::numeric, 2) AS readiness_score_2021,
            ROUND(engagement_metric_2021::numeric, 4) AS engagement_metric_2021,
            readiness_rank_desc,
            engagement_rank_asc,
            (CASE WHEN readiness_rank_desc = 1 AND engagement_rank_asc <= 3 THEN 'Yes' ELSE 'No' END) AS is_outlier
        FROM ranks
        ORDER BY readiness_score_2021 DESC;
    """
}

with engine.begin() as conn:
    for table_name, query in queries.items():
        print(f"Writing analytics.{table_name}...")
        df_res = pd.read_sql_query(query, conn)
        df_res.to_sql(table_name, conn, schema="analytics", if_exists="replace", index=False)

In [ ]:
# Define and run primary analysis queries
primary_queries = {
    "primary_analysis_q1": """
        WITH yearly_circulation AS (
            SELECT
                extract(year from month)::int AS year,
                SUM(copies_printed) AS total_copies_printed,
                SUM(copies_returned) AS total_copies_returned,
                SUM(net_circulation) AS total_net_circulation
            FROM cleaned_data.fact_print_sales
            GROUP BY 1
        ),
        yoy_changes AS (
            SELECT
                year,
                total_copies_printed,
                total_copies_returned,
                total_net_circulation,
                LAG(total_copies_printed) OVER (ORDER BY year) AS prev_printed,
                LAG(total_net_circulation) OVER (ORDER BY year) AS prev_net
            FROM yearly_circulation
        )
        SELECT
            year,
            total_copies_printed,
            total_copies_returned,
            total_net_circulation,
            ROUND(((total_copies_printed - prev_printed)::numeric / prev_printed) * 100, 2) AS pct_change_printed,
            ROUND(((total_net_circulation - prev_net)::numeric / prev_net) * 100, 2) AS pct_change_net
        FROM yoy_changes
        ORDER BY year;
    """,
    "primary_analysis_q2": """
        SELECT
            c.city AS city_name,
            SUM(p.copies_printed) AS copies_printed_2024,
            SUM(p.net_circulation) AS net_circulation_2024,
            ROUND((SUM(p.net_circulation)::numeric / SUM(SUM(p.net_circulation)) OVER ()) * 100, 2) AS circulation_contribution_pct
        FROM cleaned_data.fact_print_sales p
        JOIN cleaned_data.dim_city c ON p.city_id = c.city_id
        WHERE extract(year from p.month) = 2024
        GROUP BY c.city
        ORDER BY net_circulation_2024 DESC;
    """,
    "primary_analysis_q3": """
        SELECT
            c.city AS city_name,
            extract(year from p.month)::int AS year,
            SUM(p.copies_printed) AS total_copies_printed,
            SUM(p.copies_returned) AS total_copies_returned,
            ROUND((SUM(p.copies_returned)::numeric / SUM(p.copies_printed)) * 100, 2) AS return_waste_pct
        FROM cleaned_data.fact_print_sales p
        JOIN cleaned_data.dim_city c ON p.city_id = c.city_id
        GROUP BY c.city, year
        ORDER BY return_waste_pct DESC, year;
    """,
    "primary_analysis_q4": """
        SELECT
            cat.standard_ad_category AS category_name,
            SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2019' THEN r.ad_revenue_in_inr ELSE 0 END) AS rev_2019,
            SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2020' THEN r.ad_revenue_in_inr ELSE 0 END) AS rev_2020,
            SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2021' THEN r.ad_revenue_in_inr ELSE 0 END) AS rev_2021,
            SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2022' THEN r.ad_revenue_in_inr ELSE 0 END) AS rev_2022,
            SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2023' THEN r.ad_revenue_in_inr ELSE 0 END) AS rev_2023,
            SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2024' THEN r.ad_revenue_in_inr ELSE 0 END) AS rev_2024,
            ROUND((((SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2024' THEN r.ad_revenue_in_inr ELSE 0 END) - 
                    SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2019' THEN r.ad_revenue_in_inr ELSE 0 END))::numeric / 
                    NULLIF(SUM(CASE WHEN substring(r.quarter from 1 for 4) = '2019' THEN r.ad_revenue_in_inr ELSE 0 END), 0)) * 100)::numeric, 2) AS total_growth_pct
        FROM cleaned_data.fact_ad_revenue r
        JOIN cleaned_data.dim_ad_category cat ON r.ad_category_id = cat.ad_category_id
        GROUP BY cat.standard_ad_category
        ORDER BY total_growth_pct DESC;
    """,
    "primary_analysis_q5": """
        WITH city_2024_ad AS (
            SELECT city_id, SUM(ad_revenue_in_inr) AS ad_revenue_2024
            FROM cleaned_data.fact_ad_revenue
            WHERE quarter LIKE '2024%%'
            GROUP BY city_id
        ),
        city_2024_circ AS (
            SELECT city_id, SUM(net_circulation) AS net_circulation_2024
            FROM cleaned_data.fact_print_sales
            WHERE extract(year from month) = 2024
            GROUP BY city_id
        )
        SELECT
            c.city AS city_name,
            COALESCE(ad.ad_revenue_2024, 0) AS ad_revenue_2024,
            COALESCE(circ.net_circulation_2024, 0) AS net_circulation_2024,
            ROUND((COALESCE(ad.ad_revenue_2024, 0) / NULLIF(COALESCE(circ.net_circulation_2024, 0), 0))::numeric, 2) AS revenue_per_net_copy_2024
        FROM cleaned_data.dim_city c
        LEFT JOIN city_2024_ad ad ON c.city_id = ad.city_id
        LEFT JOIN city_2024_circ circ ON c.city_id = circ.city_id
        ORDER BY ad_revenue_2024 DESC;
    """,
    "primary_analysis_q6": """
        WITH readiness_2021 AS (
            SELECT
                city_id,
                AVG((literacy_rate + smartphone_penetration + internet_penetration) / 3.0) AS readiness_score_2021
            FROM cleaned_data.fact_city_readiness
            WHERE quarter LIKE '2021%%'
            GROUP BY city_id
        ),
        pilot_2021 AS (
            SELECT
                city_id,
                SUM(downloads_or_accesses) AS total_pilot_downloads,
                SUM(users_reached) AS total_users_reached,
                ROUND(SUM(downloads_or_accesses)::numeric / SUM(users_reached) * 100, 2) AS pilot_engagement_pct
            FROM cleaned_data.fact_digital_pilot
            WHERE launch_month LIKE '2021%%'
            GROUP BY city_id
        )
        SELECT
            c.city AS city_name,
            ROUND(r.readiness_score_2021::numeric, 2) AS digital_readiness_score_2021,
            COALESCE(p.total_pilot_downloads, 0) AS total_pilot_downloads,
            COALESCE(p.pilot_engagement_pct, 0.0) AS pilot_engagement_pct
        FROM cleaned_data.dim_city c
        JOIN readiness_2021 r ON c.city_id = r.city_id
        LEFT JOIN pilot_2021 p ON c.city_id = p.city_id
        ORDER BY digital_readiness_score_2021 DESC;
    """,
    "primary_analysis_q7": """
        WITH yearly_ad_rev AS (
            SELECT
                city_id,
                substring(quarter from 1 for 4)::int AS year,
                SUM(ad_revenue_in_inr) AS ad_revenue
            FROM cleaned_data.fact_ad_revenue
            GROUP BY city_id, year
        ),
        yearly_circ AS (
            SELECT
                city_id,
                extract(year from month)::int AS year,
                SUM(net_circulation) AS net_circulation
            FROM cleaned_data.fact_print_sales
            GROUP BY city_id, year
        )
        SELECT
            c.city AS city_name,
            ad19.year AS year_2019,
            ROUND((ad19.ad_revenue / circ19.net_circulation)::numeric, 2) AS revenue_per_copy_2019,
            ad24.year AS year_2024,
            ROUND((ad24.ad_revenue / circ24.net_circulation)::numeric, 2) AS revenue_per_copy_2024,
            ROUND((((ad24.ad_revenue / circ24.net_circulation) - (ad19.ad_revenue / circ19.net_circulation)) / (ad19.ad_revenue / circ19.net_circulation) * 100)::numeric, 2) AS roi_growth_pct
        FROM dim_city c
        JOIN yearly_ad_rev ad19 ON c.city_id = ad19.city_id AND ad19.year = 2019
        JOIN yearly_circ circ19 ON c.city_id = circ19.city_id AND circ19.year = 2019
        JOIN yearly_ad_rev ad24 ON c.city_id = ad24.city_id AND ad24.year = 2024
        JOIN yearly_circ circ24 ON c.city_id = circ24.city_id AND circ24.year = 2024
        ORDER BY revenue_per_copy_2024 DESC;
    """,
    "primary_analysis_q8": """
        WITH digital_readiness AS (
            SELECT
                city_id,
                AVG((literacy_rate + smartphone_penetration + internet_penetration) / 3.0) AS readiness_score
            FROM cleaned_data.fact_city_readiness
            WHERE quarter LIKE '2024%%'
            GROUP BY city_id
        ),
        pilot_performance AS (
            SELECT
                city_id,
                SUM(downloads_or_accesses)::numeric / SUM(users_reached) AS engagement_rate
            FROM cleaned_data.fact_digital_pilot
            GROUP BY city_id
        ),
        print_decline AS (
            SELECT
                city_id,
                ROUND((SUM(CASE WHEN extract(year from month) = 2024 THEN net_circulation ELSE 0 END) - 
                       SUM(CASE WHEN extract(year from month) = 2019 THEN net_circulation ELSE 0 END))::numeric / 
                       NULLIF(SUM(CASE WHEN extract(year from month) = 2019 THEN net_circulation ELSE 0 END), 0) * 100, 2) AS decline_pct
            FROM cleaned_data.fact_print_sales
            GROUP BY city_id
        )
        SELECT
            c.city AS city_name,
            c.tier,
            ROUND(dr.readiness_score::numeric, 2) AS digital_readiness_2024_pct,
            ROUND((pp.engagement_rate * 100)::numeric, 2) AS pilot_engagement_pct,
            pd.decline_pct AS print_decline_2019_2024_pct,
            ROUND((dr.readiness_score * pp.engagement_rate * ABS(pd.decline_pct))::numeric, 2) AS composite_relaunch_score,
            RANK() OVER (ORDER BY (dr.readiness_score * pp.engagement_rate * ABS(pd.decline_pct)) DESC) AS relaunch_rank
        FROM dim_city c
        JOIN digital_readiness dr ON c.city_id = dr.city_id
        JOIN pilot_performance pp ON c.city_id = pp.city_id
        JOIN print_decline pd ON c.city_id = pd.city_id
        ORDER BY composite_relaunch_score DESC;
    """
}

with engine.begin() as conn:
    for table_name, query in primary_queries.items():
        print(f"Writing analytics.{table_name}...")
        df_res = pd.read_sql_query(query, conn)
        df_res.to_sql(table_name, conn, schema="analytics", if_exists="replace", index=False)
print("All queries executed and loaded into analytics schema.")